# Rightmove rental listings → DataFrame

Parses Rightmove's `to-rent` search results into a tidy `pandas` DataFrame.

**How it works:** Rightmove embeds the full, structured listing data in a
`<script id="__NEXT_DATA__">` JSON blob on the search-results page. We extract
that JSON directly — no fragile HTML scraping. Every field we want
(price, availability date, address, coordinates) is already in there.

For each listing we pull:
- **Start date** — `letAvailableDate`
- **Monthly cost (pcm)** — from `price.displayPrices`
- **Location name** — `displayAddress`
- **Distance as the crow flies (km)** to three fixed places: Paddington Station,
  OC&C Strategy Consultants, and DCMS (civil service), plus a **total** of the three.

> ⚠️ **Please be a considerate scraper.** This reads Rightmove's public search
> pages. Keep request volume low (there's a polite delay between pages) and
> check their [Terms of Use](https://www.rightmove.co.uk/this-site/terms-of-use.html)
> before doing anything at scale or commercially.


## 1. Configuration — loaded from `config.yaml`

All tunable settings (search parameters, target locations, page cap and delay)
live in **`config.yaml`** next to this notebook. Edit that file and re-run this
cell to re-target the search — no need to touch the code.

In [ ]:
import time
import math
import re
import json
from pathlib import Path
from datetime import datetime, timezone

import requests
import yaml
import pandas as pd

# ---------------------------------------------------------------------------
# CONFIGURATION is loaded from an external file (config.yaml) so you can
# re-target the search without touching code.  Edit config.yaml and re-run.
# ---------------------------------------------------------------------------
CONFIG_PATH = Path("config.yaml")

with open(CONFIG_PATH) as f:
    CONFIG = yaml.safe_load(f)

SEARCH = CONFIG["search"]
TARGETS = CONFIG["targets"]
MAX_PROPERTIES = CONFIG.get("max_properties", 200)
PAGE_DELAY_SECONDS = CONFIG.get("page_delay_seconds", 1.5)

print(f"Loaded config from {CONFIG_PATH.resolve()}")
print(f"  search area: {SEARCH['locationIdentifier']}  |  {len(TARGETS)} targets  |  cap {MAX_PROPERTIES}")

# ---------------------------------------------------------------------------
# Fixed endpoints / headers — not user-tunable, so they stay in code.
# ---------------------------------------------------------------------------
RIGHTMOVE_BASE = "https://www.rightmove.co.uk"
SEARCH_URL = RIGHTMOVE_BASE + "/property-to-rent/find.html"
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36"
    )
}


## 2. Fetch + parse the `__NEXT_DATA__` JSON

In [2]:
NEXT_DATA_RE = re.compile(
    r'<script id="__NEXT_DATA__" type="application/json">(.*?)</script>', re.S
)


def build_params(search: dict, index: int = 0) -> dict:
    """Translate our friendly SEARCH dict into Rightmove query params."""
    params = {
        "locationIdentifier": search["locationIdentifier"],
        "minBedrooms": search.get("minBedrooms"),
        "maxPrice": search.get("maxPrice"),
        "minPrice": search.get("minPrice"),
        "radius": search.get("radius"),
        "channel": search.get("channel", "RENT"),
        "sortType": search.get("sortType", 6),
        "propertyTypes": search.get("propertyTypes"),
        "maxDaysSinceAdded": search.get("maxDaysSinceAdded"),
        "transactionType": "LETTING",
        "index": index,
        "includeLetAgreed": str(search.get("include_let_agreed", False)).lower(),
    }
    # drop None values so we don't send empty params
    return {k: v for k, v in params.items() if v is not None}


def fetch_page(search: dict, index: int = 0) -> dict:
    """Fetch one results page and return the parsed searchResults object."""
    resp = requests.get(
        SEARCH_URL, params=build_params(search, index), headers=HEADERS, timeout=30
    )
    resp.raise_for_status()
    m = NEXT_DATA_RE.search(resp.text)
    if not m:
        raise RuntimeError("Could not find __NEXT_DATA__ — page layout may have changed.")
    data = json.loads(m.group(1))
    return data["props"]["pageProps"]["searchResults"]


def fetch_all_properties(search: dict, max_properties: int = MAX_PROPERTIES) -> list:
    """Page through results (24 per page) up to max_properties."""
    first = fetch_page(search, index=0)
    total = first.get("resultCount") or 0
    if isinstance(total, str):
        total = int(total.replace(",", ""))
    props = list(first["properties"])
    print(f"Total results reported by Rightmove: {total}")

    index = len(props)
    while index < min(total, max_properties):
        time.sleep(PAGE_DELAY_SECONDS)
        page = fetch_page(search, index=index)
        batch = page.get("properties", [])
        if not batch:
            break
        props.extend(batch)
        index += len(batch)
        print(f"  fetched {len(props)} / {min(total, max_properties)}")

    return props[:max_properties]


## 3. Distance — haversine (as the crow flies)

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2) -> float:
    """Great-circle distance in kilometres between two lat/lon points."""
    if None in (lat1, lon1, lat2, lon2):
        return float("nan")
    R = 6371.0088  # mean Earth radius, km
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlmb = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dlmb / 2) ** 2
    return 2 * R * math.asin(math.sqrt(a))


def crow_flies_km(lat, lon, target) -> float:
    """Crow-flies distance (km) from a point to a target location."""
    return haversine_km(lat, lon, target["lat"], target["lon"])


## 4. Flatten each listing into a row

In [ ]:
def pcm_from_price(price: dict):
    """Return monthly rent (int, £pcm) from a Rightmove price object."""
    if not price:
        return None
    # Prefer the pre-formatted "pcm" display string.
    for dp in price.get("displayPrices", []):
        txt = dp.get("displayPrice", "")
        if "pcm" in txt:
            digits = re.sub(r"[^0-9]", "", txt)
            if digits:
                return int(digits)
    # Fall back to computing from amount + frequency.
    amount, freq = price.get("amount"), price.get("frequency")
    if amount is None:
        return None
    if freq == "weekly":
        return round(amount * 52 / 12)
    if freq == "monthly":
        return int(amount)
    return None


def parse_date(s):
    if not s:
        return pd.NaT
    try:
        dt = pd.to_datetime(s, utc=True).tz_localize(None)
    except Exception:
        return pd.NaT
    # Rightmove uses the 1970 epoch as a sentinel for "available now".
    if dt.year <= 1970:
        return pd.NaT
    return dt


def availability_label(s):
    """Human-readable availability: 'Now' for the epoch sentinel, else the date."""
    dt = parse_date(s)
    if pd.isna(dt):
        # distinguish genuine "available now" (epoch) from missing
        return "Now" if s else None
    return dt.date().isoformat()


def listing_to_row(p: dict) -> dict:
    loc = p.get("location") or {}
    lat, lon = loc.get("latitude"), loc.get("longitude")
    row = {
        "id": p.get("id"),
        "address": p.get("displayAddress"),
        "bedrooms": p.get("bedrooms"),
        "bathrooms": p.get("bathrooms"),
        "property_type": p.get("propertySubType"),
        "pcm": pcm_from_price(p.get("price")),
        "available_from": parse_date(p.get("letAvailableDate")),
        "available_label": availability_label(p.get("letAvailableDate")),
        "listed_on": parse_date(p.get("firstVisibleDate")),
        "added_or_reduced": p.get("addedOrReduced"),
        "agent": (p.get("customer") or {}).get("branchDisplayName"),
        "latitude": lat,
        "longitude": lon,
        "url": RIGHTMOVE_BASE + (p.get("propertyUrl") or ""),
        "summary": p.get("summary"),
    }
    # Crow-flies distance (km) to each target, plus their sum.
    total_km = 0.0
    for name, tgt in TARGETS.items():
        d = crow_flies_km(lat, lon, tgt)
        row[f"km_to_{name}"] = round(d, 2)
        total_km += d
    row["km_total"] = round(total_km, 2)
    return row


## 5. Run it → build the DataFrame

In [5]:
properties = fetch_all_properties(SEARCH)
df = pd.DataFrame([listing_to_row(p) for p in properties])
print(f"\nBuilt DataFrame with {len(df)} listings.")
df.head()


Total results reported by Rightmove: 454


  fetched 50 / 200


  fetched 75 / 200


  fetched 100 / 200


  fetched 125 / 200


  fetched 150 / 200


  fetched 175 / 200


  fetched 200 / 200

Built DataFrame with 200 listings.


,id,address,bedrooms,bathrooms,property_type,pcm,available_from,available_label,listed_on,added_or_reduced,agent,latitude,longitude,url,summary,miles_to_Paddington,miles_to_OC&C,miles_to_DCMS
0,90354630,"Quentin House, Gray Street, London, SE1",3,2.0,Apartment,3250,2026-08-26,2026-08-26,2026-07-01 12:20:58,Added on 01/07/2026,"Hello Neighbour, London",51.501140,-0.107161,https://www.rightmove.co.uk/properties/9035463...,Stunning 3 Bedroom Flat Furnished with Private...,3.10,1.16,0.83
1,89350518,"Birkdale House, London, E14",3,2.0,Flat,3150,2026-07-05,2026-07-05,2026-06-05 14:13:08,Reduced today,"OpenRent, London",51.513150,-0.029403,https://www.rightmove.co.uk/properties/8935051...,Spacious 3 Bed / 2 Bath Apartment with Huge Pr...,6.28,3.37,4.20
2,90509076,"Rutherford House, 483 Battersea Park Road, Bat...",3,2.0,Flat,3500,2027-08-04,2027-08-04,2026-07-05 02:52:20,Added today,"John D Wood & Co. Lettings, Battersea",51.471749,-0.163986,https://www.rightmove.co.uk/properties/9050907...,We are pleased to present this large three bed...,3.06,4.01,2.77
3,89612742,"Edison Building, Westferry Road, Canary Wharf,...",3,2.0,Apartment,3250,2026-08-23,2026-08-23,2026-06-11 21:49:11,Reduced yesterday,"Reiss Samuels, London",51.500186,-0.026214,https://www.rightmove.co.uk/properties/8961274...,Students welcome. Superb location a short walk...,6.51,3.70,4.30
4,90501351,"Hillfield House, Canonbury",3,1.0,Flat,3000,2026-07-18,2026-07-18,2026-07-04 17:26:40,Added yesterday,"London Move, London",51.549370,-0.092600,https://www.rightmove.co.uk/properties/9050135...,A spacious three-bedroom split-level garden ma...,4.27,2.27,3.44


In [ ]:
# A focused view of exactly the columns you asked about
view_cols = (
    ["address", "pcm", "available_label"]
    + [f"km_to_{name}" for name in TARGETS]
    + ["km_total", "url"]
)
df.sort_values("available_from")[view_cols]


## 6. Future work — placeholders

### 6a. Real travel time via TfL ✅ (implemented in `travel_utils.py`)
Door-to-door public-transport journey time using the **TfL Journey Planner API**
(free, no API key) now lives in **`travel_utils.py`**, with a throwaway test notebook
in **`test_travel_utils.ipynb`**. It models real London journeys (tube/bus/rail/walking).

```python
# from travel_utils import travel_times_to_targets, next_weekday_at
#
# times = travel_times_to_targets(
#     (lat, lon), TARGETS,
#     arrival_time=next_weekday_at(9),   # arrive by 09:00 next weekday
# )
# -> {"Paddington": {"minutes": 21, "distance_km": 5.4,
#                    "text": "21 min (walking → tube → walking)", ...}, ...}
```
Free and keyless for low volume; set a `TFL_APP_KEY` env var (free from
api-portal.tfl.gov.uk) for higher rate limits. One request per (listing, target) —
cache results if re-running often. London-only, which suits these central-London targets.

**Not yet wired into the DataFrame above** — `listing_to_row` still uses crow-flies km.
Fold `travel_times_to_targets` into it when you're ready (add a polite delay + cache
across the full listing set).

### 6b. Deal-breaker flagging
Flag listings against a list of deal-breaker attributes (keywords in
`summary` / `keyFeatures`, price bands, min bedrooms/bathrooms, max distance, …).

```python
# DEAL_BREAKERS = {
#     "no_ground_floor":  lambda p: "ground floor" in (p.get("summary") or "").lower(),
#     "too_far_from_DCMS": lambda row: row["km_to_DCMS"] > 8,
#     "share_required":   lambda p: "house share" in (p.get("summary") or "").lower(),
# }
# -> add a `flags` column listing which deal-breakers each listing trips.
```
